# Q14 — Transfer the unchanged Q12 learned strategy to Q13's memory model

**Upload only this notebook to Jupyter on your server. All source code is embedded.**
It uses your existing P1 NetSquid/Qoala environment and the completed Q12/Q13 directories.
No package installation or NetSquid login is required if that environment is still installed.

**First run the Q12 and Q13 raw-audit notebooks and obtain PASS for both.**
This notebook finds their default audit folders automatically and verifies the exact audited
data/source identities. If either audit is missing or failed, it stops before new simulations.
The prerequisite folders are already filled in below. If an auditor used a custom output folder,
set the corresponding `Q12_AUDIT_DIR` or `Q13_AUDIT_DIR` to that printed path.

Then choose **Run All**. Defaults are `MODE="full"`, `WORKERS=8`, `RESUME_RUN_DIR="auto"`.
Use all eight workers only when eight allocated cores are free. Each worker uses one numerical-library
thread. If you are also running another experiment, lower `WORKERS` to the available allocation.

The notebook creates a separate `q14-...` result folder. Original Q12/Q13 results and audit reports
are read-only inputs. The default full run proceeds automatically from correctness pilot to evaluation
and export. It prints the run directory immediately and a rough runtime estimate after the pilot.

| Stage | Executions | Purpose |
|---|---:|---|
| Correctness pilot | 1,944 | 1,176 noise-free controls; all 24 teleportation state/Bell combinations per arm; 768 noisy checks spanning every source campaign and both memory profiles |
| Evaluation | 98,304 | 2 workloads × 2 profiles × 32 source campaigns × 3 arms × 256 fresh trials |

The pilot tests correctness, not whether the learned strategy is effective. It must pass before evaluation.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.home() / "P1"
MODE = "full"
WORKERS = 8  # Uses your eight allocated cores; lower this if other jobs use them.
RESUME_RUN_DIR = "auto"

Q12_RUN_DIR = PROJECT_DIR / "results" / "q12-20261013T034634906569Z"
Q13_RUN_DIR = PROJECT_DIR / "results" / "q13-20261013T045651939437Z"

# None locates the matching raw-audit outputs made by the provided notebooks.
# If you customized an auditor's AUDIT_DIR, put that printed folder here.
Q12_AUDIT_DIR = None
Q13_AUDIT_DIR = None

PYTHON = None  # Reuse the existing P1 interpreter; no package installation.
REPO = None    # Reuse the existing pinned Qoala checkout.
EXPORT_RAW = False


In [ ]:
import base64, fcntl, gzip, hashlib, json, os, sys

SOURCE_BUNDLE_SHA256 = 'a418e488b504d50ab4fef19f7962d672f2cdf6925da92b25e944f8a78d201f59'
_EMBEDDED = '...'
_EXPECTED_FILES = ['q14.py', 'q14_design.py', 'q14_engine.py', 'q14_analysis.py', 'q14_prerequisites.py', 'q14_tests.py', 'synthetic_records.py', 'p1_engine.py', 'q6_reference.py', 'time_checks.py', 'hardware_models.json', 'q13_design.py', 'q13_raw_checks.py', 'parent_sources.json', 'q14_expected_prerequisites.json']

def _sha_bytes(value):
    return hashlib.sha256(value).hexdigest()

def install_embedded_code(project):
    data = gzip.decompress(base64.b64decode(_EMBEDDED))
    if _sha_bytes(data) != SOURCE_BUNDLE_SHA256:
        raise RuntimeError("Embedded source bundle checksum mismatch")
    files = json.loads(data)
    if set(files) != set(_EXPECTED_FILES):
        raise RuntimeError("Unexpected embedded source files")
    base = Path(project).expanduser().resolve() / "experiment_code"
    for original in (Q12_RUN_DIR, Q13_RUN_DIR, Q12_AUDIT_DIR, Q13_AUDIT_DIR):
        if original is not None:
            original = Path(original).expanduser().resolve()
            if base == original or original in base.parents or base in original.parents:
                raise RuntimeError("Q14 code must be installed outside all original result and audit folders")
    base.mkdir(parents=True, exist_ok=True)
    folder = base / ("q14-" + SOURCE_BUNDLE_SHA256[:16])
    with (base / (folder.name + ".install.lock")).open("a+") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        folder.mkdir(exist_ok=True)
        for name, content in files.items():
            if Path(name).name != name:
                raise RuntimeError("Invalid embedded filename")
            expected = content.encode()
            destination = folder / name
            if destination.exists():
                if destination.read_bytes() != expected:
                    raise RuntimeError("Installed source differs: " + str(destination) +
                                       ". Preserve it; do not overwrite or mix versions.")
            else:
                temporary = destination.with_name(name + ".tmp-" + str(os.getpid()))
                temporary.write_bytes(expected)
                os.replace(temporary, destination)
            if _sha_bytes(destination.read_bytes()) != _sha_bytes(expected):
                raise RuntimeError("Source verification failed: " + name)
    return folder

PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
CODE_DIR = install_embedded_code(PROJECT_DIR)
print("Embedded code installed and verified:", CODE_DIR, flush=True)
print("Source bundle hash:", SOURCE_BUNDLE_SHA256, flush=True)


In [ ]:
import shutil, subprocess

if MODE not in ("pilot", "full"):
    raise ValueError('MODE must be "pilot" or "full"')
if not isinstance(WORKERS, int) or isinstance(WORKERS, bool) or WORKERS < 1:
    raise ValueError("WORKERS must be a positive integer")
available_cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
if available_cpus and WORKERS > available_cpus:
    raise ValueError(f"WORKERS={WORKERS}, but this process can use only {available_cpus} CPUs")

RUN_ENV = os.environ.copy()
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                 "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    RUN_ENV[variable] = "1"
RUN_ENV["PYTHONUNBUFFERED"] = "1"
RUN_ENV["PYTHONDONTWRITEBYTECODE"] = "1"
RUN_ENV.pop("PYTHONOPTIMIZE", None)

# Resolve the exact auditor-version paths without guessing among result folders.
parent_expectations = json.loads((CODE_DIR / "q14_expected_prerequisites.json").read_text())
Q12_RUN_DIR = Path(Q12_RUN_DIR).expanduser().resolve()
Q13_RUN_DIR = Path(Q13_RUN_DIR).expanduser().resolve()
if Q12_AUDIT_DIR is None:
    Q12_AUDIT_DIR = PROJECT_DIR / "audits" / (Q12_RUN_DIR.name + "-raw-" + parent_expectations["q12"]["audit_bundle_sha256"][:12])
if Q13_AUDIT_DIR is None:
    Q13_AUDIT_DIR = PROJECT_DIR / "audits" / (Q13_RUN_DIR.name + "-raw-" + parent_expectations["q13"]["audit_bundle_sha256"][:12])
Q12_AUDIT_DIR = Path(Q12_AUDIT_DIR).expanduser().resolve()
Q13_AUDIT_DIR = Path(Q13_AUDIT_DIR).expanduser().resolve()
for label, folder in (("Q12", Q12_AUDIT_DIR), ("Q13", Q13_AUDIT_DIR)):
    print(label + " raw audit:", folder, flush=True)
    if not (folder / "AUDIT_COMPLETE.json").is_file():
        raise RuntimeError("Run the " + label + " raw-audit notebook first. Q14 has not started. "
                           "If you used a custom AUDIT_DIR, put it in " + label + "_AUDIT_DIR above.")

repo_candidates = ([Path(REPO).expanduser()] if REPO is not None else
                   [PROJECT_DIR / name for name in ("qoala-sim", "p1-qoala", "qoala")] +
                   [Path.home() / "p1-qoala", Path.home() / "qoala-sim"])
repos = []
for candidate in repo_candidates:
    if not candidate.is_dir():
        continue
    result = subprocess.run(["git", "-C", str(candidate), "rev-parse", "HEAD"],
                            capture_output=True, text=True)
    if result.returncode == 0 and result.stdout.strip() == "31fa2c9699578d227736557f35cd01a134c4bbde":
        repos.append(candidate.resolve())
if not repos:
    raise RuntimeError("Could not locate the existing pinned Qoala checkout. Set REPO in the configuration "
                       "cell to its directory, then rerun. Checked: " + ", ".join(map(str, repo_candidates)))
REPO = repos[0]

if PYTHON is not None:
    located = shutil.which(str(PYTHON))
    python_candidates = [located or str(Path(PYTHON).expanduser().absolute())]
else:
    # Do not resolve interpreter symlinks: doing so can bypass a virtual environment.
    python_candidates = [str(PROJECT_DIR / ".venv-p1/bin/python"),
                         str(PROJECT_DIR / ".venv/bin/python"), sys.executable]
python_candidates = list(dict.fromkeys(python_candidates))
preflight = "import json,sys; import q6_reference; info=q6_reference.check_repo(sys.argv[1]); " \
            "import scipy; info['scipy']=scipy.__version__; print(json.dumps(info,indent=2))"
failures = []
chosen = None
for candidate in python_candidates:
    if not Path(candidate).is_file():
        continue
    result = subprocess.run([candidate, "-c", preflight, str(REPO)], cwd=CODE_DIR,
                            env=RUN_ENV, capture_output=True, text=True)
    if result.returncode == 0:
        chosen = candidate
        print("Using Python:", candidate, flush=True)
        print("Using Qoala:", REPO, flush=True)
        print(result.stdout, flush=True)
        break
    failures.append(candidate + "\n" + result.stdout + result.stderr)
if chosen is None:
    print("\n".join(failures), flush=True)
    raise RuntimeError("No validated environment found. Set PYTHON to the same interpreter used for "
                       "Q11/Q9S. Required: Python 3.10, NetSquid 1.1.8, pydynaa 1.0.2, Qoala 2.1.0, "
                       "NumPy 1.26.4, netqasm 2.3.0 and SciPy. This notebook has not changed your environment.")
PYTHON = chosen

# Capture and print so output appears reliably in Jupyter as well as a terminal.
tests = subprocess.run([PYTHON, "-u", str(CODE_DIR / "q14_tests.py")],
                       cwd=CODE_DIR, env=RUN_ENV, capture_output=True, text=True)
print(tests.stdout, end="", flush=True)
if tests.stderr:
    print(tests.stderr, end="", flush=True)
if tests.returncode:
    raise RuntimeError("Local design checks failed; do not start the experiment")
parent_command = [PYTHON, "-u", str(CODE_DIR / "q14_prerequisites.py"),
    "--q12-run", str(Q12_RUN_DIR), "--q13-run", str(Q13_RUN_DIR),
    "--q12-audit", str(Q12_AUDIT_DIR), "--q13-audit", str(Q13_AUDIT_DIR)]
parents = subprocess.run(parent_command, cwd=CODE_DIR, env=RUN_ENV, capture_output=True, text=True)
print(parents.stdout, end="", flush=True)
if parents.stderr:
    print(parents.stderr, end="", flush=True)
if parents.returncode:
    raise RuntimeError("Parent audit verification failed. Q14 has not started; share the diagnostic above.")
print(f"Preflight passed. Mode: {MODE} | workers: {WORKERS}. Next: 1,944 pilot executions, then 98,304 evaluation executions.", flush=True)


## Frozen research question and protocol

**Does the strategy already learned in Q12 still increase expected infidelity over equal-work random
timing under the two Q13 memory parameterizations?**

- Transfer **all 32 learned phases for each workload**, unchanged. No new calibration, campaign selection,
  phase search, or adjustment after observing Q14 outcomes. The policy receives the supplied coarse epoch
  and its frozen phase, never the actual victim start, outcome, state or trace.
- Arms are **no competitor**, **uniform random phase**, and **transferred learned phase**. Every competing
  arm submits four ordinary 30-instruction CPU jobs, totaling **1.2 ms** of service.
- Both workloads and both profiles are retained. Victim requests vary within **±0.5 ms** of the epoch;
  one-way messages vary uniformly from 0.5 to 1.5 ms. Inputs use new Q14 seed roots and are paired across
  arms/profiles. No Q12/Q13 evaluation record is reused as a Q14 observation.
- Native T1/T2 noise stays disabled. The unchanged Q13 full-age isotropic channel is applied once before
  verification. Ideal local gates/EPR and existing P1 CPU/QNOS/link timing assumptions are retained.
- Primary endpoint: **learned minus random expected infidelity** before verification, four contrasts.
  Report actual sampled errors, unfinished outputs, storage, response times and calibration costs as well.

Each primary contrast uses **32 source calibration campaigns**, with 256 fresh paired trials per arm.
20,000 bootstrap resamples of campaign means produce approximate 95% simultaneous intervals with
Bonferroni allocation over the four primary contrasts. This treats Q12's independent calibration
campaigns as replications of the original learning procedure, including variation in learned phases.
It is not a conditional test-set interval for one fixed phase. Separate exact sign tests assume
independent, exchangeable source campaigns and concern direction rather than mean effect.
Observed-error intervals are exploratory pointwise intervals. Physical parameter uncertainty is not included.

Per-workload support requires both profile-specific primary lower bounds above zero, with every compared
execution complete and a fidelity available. Broad support requires all four. Otherwise report the
workload-specific, inconclusive or negative result. **No positive result is guaranteed or required to finish.**
Unfinished executions remain in the offered denominator; any such compared execution suppresses the
primary fidelity-effect inference and triggers worst-case loss bounds. There is no latency filter.

**Inherited cost remains visible.** Q14 performs zero new calibration, but each original Q12 campaign cost
4,608 own-application executions and 17,408 competing CPU jobs, or 5.2224 seconds of competing CPU service.
Teleportation additionally consumed 4,608 calibration EPR pairs. These are reused costs, not free learning.

This is a follow-up motivated by already inspected Q12/Q13 results. Its plan is frozen before Q14 execution;
it is not described as preregistered before those earlier results. The two profiles come from one hardware
source. Shared placement, ordinary-job admission and coarse-epoch access remain assumptions. This does not
establish deployment practicality, a complete hardware calibration, nonideal-gate/EPR transfer or sustained-load behavior.


## Run or resume

The next cell launches the pilot and, with `MODE="full"`, continues automatically to evaluation.
It prints `RUN DIRECTORY`, progress, worker RAM and a pilot-based runtime estimate.
Earlier `local tests passed` messages do not mean that NetSquid has run.

If the session ends, reopen this same notebook and **Run All with `RESUME_RUN_DIR="auto"`**.
Completed 128-record batches are hash-verified and reused; only incomplete batches rerun.
You may change `WORKERS` on resume. To resume explicitly, use the printed `q14-...` path.
To deliberately start a separate run, use `RESUME_RUN_DIR=None`.
Do not change protocol/source files or delete metadata to bypass a mismatch.

If a runtime check fails, run the final export cell manually and share the ZIP including logs.
The notebook rejects a second coordinator for the same run directory.


In [ ]:
from datetime import datetime, timezone
import signal, time

RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
POINTER = RESULTS_DIR / (".q14-latest-" + SOURCE_BUNDLE_SHA256[:16] + ".json")

def choose_run_directory(resume):
    if resume == "auto" and POINTER.exists():
        previous = json.loads(POINTER.read_text())
        if previous["source_bundle_sha256"] != SOURCE_BUNDLE_SHA256:
            raise RuntimeError("Resume pointer belongs to a different source bundle")
        root = Path(previous["run_dir"]).expanduser().resolve()
        if not root.is_dir():
            raise RuntimeError("The saved run folder is missing: " + str(root) +
                               ". Restore it, or set RESUME_RUN_DIR=None to explicitly start anew.")
        return root
    if resume is None or resume == "auto":
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
        return RESULTS_DIR / ("q14-" + stamp)
    root = Path(resume).expanduser().resolve()
    if not root.is_dir():
        raise RuntimeError("Resume folder does not exist: " + str(root))
    if not root.name.startswith("q14-"):
        raise RuntimeError("Expected a Q14 run folder; do not reuse a Q12/Q11/Q9S folder")
    return root

RUN_DIR = choose_run_directory(RESUME_RUN_DIR)
for protected in (REPO, CODE_DIR, Q12_RUN_DIR, Q13_RUN_DIR, Q12_AUDIT_DIR, Q13_AUDIT_DIR):
    if RUN_DIR == protected or protected in RUN_DIR.parents or RUN_DIR in protected.parents:
        raise RuntimeError("Q14 results must be separate from all parent, source and audit folders")
RUN_DIR.mkdir(parents=True, exist_ok=True)
pointer_temp = POINTER.with_name(POINTER.name + ".tmp-" + str(os.getpid()))
pointer_temp.write_text(json.dumps(dict(run_dir=str(RUN_DIR), source_bundle_sha256=SOURCE_BUNDLE_SHA256), indent=2))
os.replace(pointer_temp, POINTER)

print("RUN DIRECTORY:", RUN_DIR, flush=True)
print("For an explicit resume, set RESUME_RUN_DIR =", repr(str(RUN_DIR)), flush=True)
print('Keep MODE="full" to continue both stages. The default "auto" remembers this path.', flush=True)
launcher_log = RUN_DIR / ("launcher-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".log")
command = [PYTHON, "-u", str(CODE_DIR / "q14.py"), "run", "--repo", str(REPO),
           "--run-dir", str(RUN_DIR), "--mode", MODE, "--workers", str(WORKERS),
           "--q12-run", str(Q12_RUN_DIR), "--q13-run", str(Q13_RUN_DIR),
           "--q12-audit", str(Q12_AUDIT_DIR), "--q13-audit", str(Q13_AUDIT_DIR)]

with launcher_log.open("w") as handle:
    process = subprocess.Popen(command, cwd=CODE_DIR, env=RUN_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    print("Coordinator PID:", process.pid, "| log:", launcher_log, flush=True)
    try:
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            print(line, end="", flush=True)
        exit_code = process.wait()
    except BaseException:
        # Stop the coordinator and its worker subprocesses together on interruption.
        for sig in (signal.SIGINT, signal.SIGTERM, signal.SIGKILL):
            if process.poll() is not None:
                break
            try:
                os.killpg(process.pid, sig)
            except ProcessLookupError:
                break
            try:
                process.wait(timeout=10)
                break
            except subprocess.TimeoutExpired:
                pass
        print("Stopped. Completed checkpoints remain reusable in", RUN_DIR, flush=True)
        raise
    finally:
        process.stdout.close()
if exit_code:
    raise RuntimeError("Q14 stopped with exit code " + str(exit_code) +
                       ". Run the export cell below manually and share the review ZIP, including logs.")
print("Requested stages finished. The next cell displays available results and exports them.", flush=True)


## Results and export

This cell runs automatically after success. After an error, run it manually.
It exports compact outcomes, source, parent-audit evidence, immutable phase selections,
checkpoint manifests, analysis and diagnostic logs. Raw traces remain on the server by default.
Download the **review ZIP** and upload it in our chat. Keep the original Q14 folder for resuming and audit.

The expected-infidelity result and sampled verification-error result are different endpoints.
Do not assign the primary significance level to the observed-error counts. The 0.5 percentage-point
line retained in the analysis is a disclosed reporting reference, not a practical-impact threshold.


In [ ]:
import html, zipfile
from IPython.display import display, HTML, FileLink

if "RUN_DIR" not in globals():
    saved_pointer = PROJECT_DIR / "results" / (".q14-latest-" + SOURCE_BUNDLE_SHA256[:16] + ".json")
    if RESUME_RUN_DIR not in (None, "auto"):
        RUN_DIR = Path(RESUME_RUN_DIR).expanduser().resolve()
    elif saved_pointer.exists():
        RUN_DIR = Path(json.loads(saved_pointer.read_text())["run_dir"])
    else:
        raise RuntimeError("No Q14 run folder is available yet; run the preceding cells first")

print("RUN DIRECTORY:", RUN_DIR, flush=True)
for label, marker in [("Integration pilot complete", "PILOT_COMPLETE.json"),
                      ("Entire study execution complete", "COMPLETE.json")]:
    print(label + ":", (RUN_DIR / marker).exists(), flush=True)

if (RUN_DIR / "analysis.json").exists():
    report = json.loads((RUN_DIR / "analysis.json").read_text())
    headings = ["Workload", "Memory profile", "Learned − random infidelity (pp)",
                "Adjusted interval (pp)", "All targets finished", "Interval above zero"]
    table_rows = []
    for row in report["primary_contrasts"]:
        interval = row["approximate_adjusted_interval_pp"]
        effect = row["mean_difference_pp"]
        table_rows.append([row["workload"], row["profile"], "Unavailable" if effect is None else f"{effect:.4f}",
                           "Unavailable" if interval is None else f"[{interval[0]:.4f}, {interval[1]:.4f}]",
                           row["all_compared_targets_completed"], row["interval_above_zero"]])
    markup = '<table><thead><tr>' + ''.join('<th>' + html.escape(x) + '</th>' for x in headings) + '</tr></thead><tbody>'
    for row in table_rows:
        markup += '<tr>' + ''.join('<td>' + html.escape(str(x)) + '</td>' for x in row) + '</tr>'
    display(HTML(markup + '</tbody></table>'))
    print("All four primary intervals above zero:", report["all_four_primary_intervals_above_zero"], flush=True)
    print("Transfer support by workload:", report["per_workload_transfer_support"], flush=True)
    print("Full effect sizes, sampled errors, unfinished counts and inherited costs are in analysis.json.", flush=True)
    print("This tests unchanged learned-policy transfer under the declared model and epoch assumptions.", flush=True)
else:
    print("Analysis is not available yet. Exporting available checkpoints and diagnostics.", flush=True)

# Export is read-only with respect to the scientific run folder.
print("Collecting export file list...", flush=True)
all_paths = sorted(p for p in RUN_DIR.rglob("*") if p.is_file() and p.name != "RUNNING.lock")
inventory = []
paths = []
for path in all_paths:
    is_raw = path.name.endswith(".raw.jsonl.gz")
    included = bool(EXPORT_RAW or not is_raw)
    inventory.append(dict(path=str(path.relative_to(RUN_DIR)), bytes=path.stat().st_size,
                          included=included, raw_trace=is_raw))
    if included:
        paths.append(path)
byte_count = sum(item["bytes"] for item in inventory if item["included"])
raw_excluded = sum(item["raw_trace"] and not item["included"] for item in inventory)
flavor = "full" if EXPORT_RAW else "review"
archive_name = "P1_Qoala_Q14_" + RUN_DIR.name + "_" + flavor + ".zip"
ARCHIVE = PROJECT_DIR / archive_name
print(f"Exporting {len(paths):,} files ({byte_count / 1024**2:.1f} MiB before ZIP). "
      f"Raw trace files retained on server and omitted here: {raw_excluded:,}", flush=True)
temporary_archive = ARCHIVE.with_name(ARCHIVE.name + ".partial-" + str(os.getpid()))
started_export = time.monotonic()
last_export_print = started_export
with zipfile.ZipFile(temporary_archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as archive:
    for index, path in enumerate(paths, 1):
        compression = zipfile.ZIP_STORED if path.suffix == ".gz" else zipfile.ZIP_DEFLATED
        archive.write(path, arcname=str(Path(RUN_DIR.name) / path.relative_to(RUN_DIR)), compress_type=compression)
        if index == 1 or index % 100 == 0 or time.monotonic() - last_export_print >= 10 or index == len(paths):
            print(f"Archived {index:,}/{len(paths):,}", flush=True)
            last_export_print = time.monotonic()
    archive.writestr("EXPORT_MANIFEST.json", json.dumps(dict(
        run_dir=str(RUN_DIR), archive_kind=flavor, source_bundle_sha256=SOURCE_BUNDLE_SHA256,
        raw_trace_files_excluded=raw_excluded,
        note="Review archives cannot independently verify omitted raw traces. Keep original raw files for audit.",
        files=inventory), indent=2))
os.replace(temporary_archive, ARCHIVE)
print(f"Results ZIP: {ARCHIVE} ({ARCHIVE.stat().st_size / 1024**2:.1f} MiB)", flush=True)
download_folder = Path.cwd().resolve()
protected = (RUN_DIR, Q12_RUN_DIR, Q13_RUN_DIR, Q12_AUDIT_DIR, Q13_AUDIT_DIR, CODE_DIR, REPO)
if any(download_folder == p or p in download_folder.parents for p in protected):
    print("Download the ZIP from the persistent path above; the notebook is inside a protected folder.", flush=True)
else:
    download = download_folder / archive_name
    if download.resolve() != ARCHIVE.resolve():
        print("Copying archive into the notebook working directory for download...", flush=True)
        shutil.copy2(ARCHIVE, download)
    display(FileLink(archive_name))
print("Share the review ZIP first. Keep the original run directory for resume and full trace audit.", flush=True)


## Validation and scope

The embedded tests check the complete grid, paired inputs, information boundary, policy retention,
noise-free coverage, synthetic raw-record validation, known statistical effects, missing-output handling,
checkpoint corruption/resume behavior, prerequisite audit failures and unchanged parent modules.
**Local tests use synthetic data. NetSquid runtime has not been executed in the development environment.**
The server pilot provides the required integration test before the full evaluation.

The source memory parameters and their provenance are carried forward unchanged in `hardware_models.json`.
Their original baseline is checked by Q13 and its independent raw audit. Q14 tests transfer of the learned
policy into that same runtime model; it does not refit the model or repeat the published hardware experiment.
